<a href="https://colab.research.google.com/github/EunjeLee0812/Sanhak/blob/seowonryeol/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 라이브러리 설치
!pip install faster-whisper rapidfuzz g2pk konlpy python-mecab-ko

In [47]:
import gc, sys
import os, re, json, glob, csv, random, glob, time
from dataclasses import dataclass
from typing import Dict, List, Optional, Any, Tuple
from g2pk import G2p
from faster_whisper import WhisperModel
from rapidfuzz.distance import Levenshtein
from rapidfuzz import process, fuzz
from mecab import MeCab
import importlib

In [ ]:
# #Googledrive 마운트(Colab 사이트 사용 시 주석 해제)
# from google.colab import drive
# drive.mount('/content/drive')

#모듈파일 변경 후 세션 재시작 안 해도 되게 하는 코드
%load_ext autoreload
%autoreload 2

In [49]:
# 1. 파일들이 위치한 경로로 이동 colab용 
# BASE_PATH = "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/"
# 프로젝트 루트로 이동 (Colab)
# %cd /content/drive/MyDrive/0208

# 모듈 Import
from config.settings import *
from utils.normalizer import TextNormalizer
from utils.data_loader import load_transcripts
from utils.metrics import calculate_cer, calculate_wer, evaluate_proper_nouns
from core.asr_engine import ASR
from core.bias_manager import BiasManager
from core.post_processor import postprocess_with_hotwords


In [ ]:
# #그래픽카드 메모리 남용을 막기 위한 캐시 초기화

# gc.collect()
# torch.cuda.empty_cache()

# 1-5. 결과 저장 경로[현재 시간 반영해서 파일별 구분 용이]
#results 폴더 없으면 생성
import os
from config.settings import RESULTS_DIR

import random #추가
random.seed(42)

def save_results_with_summary(rows: List[Dict[str, Any]], output_path: str):
    if not rows:
        print("[WARN] 저장할 데이터가 없습니다.")
        return

    # 1. 상세 데이터 저장 (Detail)
    # --------------------------------------------------------------------------
    # 상세 데이터에 있는 키들만 추출하여 헤더로 사용
    detail_fieldnames = list(rows[0].keys())
    
    with open(output_path, "w", newline="", encoding="utf-8-sig") as f:
        # [Writer 1] 상세 데이터용 Writer 생성
        w_detail = csv.DictWriter(f, fieldnames=detail_fieldnames)
        w_detail.writeheader()
        w_detail.writerows(rows)
        
        # 2. 요약 데이터 집계 (Aggregation)
        # ----------------------------------------------------------------------
        agg = {} 

        for r in rows:
            # 그룹화 키 생성
            hotwords_key = str(r["hotwords"])
            
            key = (
                r["top_k"],
                r["postprocess_on"],
                r["hotwords_strategy"],
                r["bias_weight_update_cnt"],
                hotwords_key, 
            )

            if key not in agg:
                agg[key] = {
                    "count": 0, 
                    # "bias_weight_update_cnt": 0, 
                    "hotwords":"",
                    "total_wrong_char_cnt": 0, "total_char_cnt": 0,
                    "total_wrong_morph_cnt": 0, "total_morph_cnt": 0,
                    "pn_recall_sum": 0.0, "pn_cer_sum": 0.0, "pn_count": 0
                }

            # 데이터 누적
            g = agg[key]
            g["count"] += 1
            g["total_wrong_char_cnt"] += float(r.get("wrong_char_cnt", 0))
            g["total_char_cnt"] += float(r.get("char_cnt", 0))
            g["total_wrong_morph_cnt"] += float(r.get("wrong_morph_cnt", 0)) # 변수명 확인 (wrong_morpheme_cnt)
            g["total_morph_cnt"] += float(r.get("morph_cnt", 0))          # 변수명 확인 (morpheme_cnt)
            # g["bias_weight_update_cnt"]=r.get("bias_weight_update_cnt",0)
            g["hotwords"]=r.get("hotwords","")

            # PN 지표는 값이 있는 경우(None이 아닌 경우)에만 합산
            if r.get("pn_recall") is not None:
                g["pn_recall_sum"] += float(r["pn_recall"])
                g["pn_cer_sum"] += float(r["pn_cer"])
                g["pn_count"] += 1

        # 3. 요약 데이터 리스트 생성
        # ----------------------------------------------------------------------
        summary_rows = []
        sorted_keys = sorted(agg.keys())

        for key in sorted_keys:
            top_k, pp_on, strat, bias_cnt, hotwords= key
            stats = agg[key]
            
            # bias_cnt=stats["bias_weight_update_cnt"]
            # hw_str=stats["hotwords"]

            # Global Average 계산
            global_cer = stats["total_wrong_char_cnt"] / stats["total_char_cnt"] if stats["total_char_cnt"] > 0 else 0.0
            global_wer = stats["total_wrong_morph_cnt"] / stats["total_morph_cnt"] if stats["total_morph_cnt"] > 0 else 0.0
            
            # PN 지표 (Macro Average)
            avg_pn_recall = stats["pn_recall_sum"] / stats["pn_count"] if stats["pn_count"] > 0 else 0.0
            avg_pn_cer = stats["pn_cer_sum"] / stats["pn_count"] if stats["pn_count"] > 0 else 0.0
            
            summary_row = {
                "file": "Total_Average", # 파일명 대신 요약임을 표시
                "top_k": top_k,
                "postprocess_on": pp_on,
                "hotwords_strategy": strat,
                "bias_weight_update_cnt": bias_cnt,
                "hotwords": hotwords,
                
                "cer": f"{global_cer:.4f}",
                "wer": f"{global_wer:.4f}",
                "pn_recall": f"{avg_pn_recall:.4f}",
                "pn_cer": f"{avg_pn_cer:.4f}",
                
                # [중요] Summary에만 존재하는 필드들
                "replog": f"Total Files: {stats['count']}",
                "total_wrong_char_cnt": stats["total_wrong_char_cnt"],
                "total_char_cnt": stats["total_char_cnt"],
                "total_wrong_morph_cnt": stats["total_wrong_morph_cnt"],
                "total_morph_cnt": stats["total_morph_cnt"]
            }
            summary_rows.append(summary_row)

        # 4. 요약 데이터 저장 (새로운 Writer 사용)
        # ----------------------------------------------------------------------
        if summary_rows:
            f.write("\n") # 상세 데이터와 구분하기 위해 빈 줄 추가
            
            # [핵심 수정] 요약 데이터용 fieldnames를 새로 정의
            # 기존 detail 필드 + 새로 추가된 집계 필드들
            summary_fieldnames = [
                "file", "top_k", "postprocess_on", "hotwords_strategy", 
                "bias_weight_update_cnt", "hotwords", 
                "cer", "wer", "pn_recall", "pn_cer", "replog",
                # 상세 데이터엔 없던 새로운 필드들 추가
                "total_wrong_char_cnt", "total_char_cnt", 
                "total_wrong_morph_cnt", "total_morph_cnt"
            ]
            
            # [Writer 2] 요약 데이터용 Writer 생성
            w_summary = csv.DictWriter(f, fieldnames=summary_fieldnames)
            w_summary.writeheader() # 요약용 헤더를 다시 씀 (구분 명확화)
            w_summary.writerows(summary_rows)

    print(f"[SUCCESS] 상세 및 요약 결과가 {output_path}에 저장되었습니다.")

def summarize(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # key는 4개이므로 Tuple[int, int, str, int]
    agg: Dict[Tuple[int, int, str, int], Dict[str, Any]] = {}

    for r in rows:
        hotwords_strategy = r.get("hotwords_strategy", "random")
        bias_weight_update_cnt = int(r.get("bias_weight_update_cnt", 0))

        key = (
            int(r["top_k"]),
            int(r["postprocess_on"]),   # 🔥 postprocess_on → postprocess_on
            hotwords_strategy,
            bias_weight_update_cnt,
        )

        a = agg.setdefault(
            key,
            {
                "top_k": key[0],
                "postprocess_on": key[1],
                "hotwords_strategy": key[2],
                "bias_weight_update_cnt": key[3],
                "files_num": 0,
                "global_cer": 0.0,
                "global_wer": 0.0,
                "pn_recall_sum": 0.0,
                "pn_cer_sum": 0.0,
                "pn_count": 0, # 추가
                "total_wrong_char_cnt":0,
                "total_char_cnt":0, 
                "total_wrong_morph_cnt":0, 
                "total_morph_cnt":0
            },
        )

        #파일 개수 및 전체 cer, wer 계산
        a["files_num"] += 1
        a["total_wrong_char_cnt"] += float(r["wrong_char_cnt"])
        a["total_wrong_morph_cnt"] += float(r["wrong_morph_cnt"])
        a["total_char_cnt"]+=r["char_cnt"]
        a["total_morph_cnt"]+=r["morph_cnt"]

        if r.get("pn_recall") is not None:
            a["pn_recall_sum"] += float(r["pn_recall"])
            a["pn_cer_sum"] += float(r["pn_cer"])
            a["pn_count"] += 1

    out = []
    for _, a in sorted(agg.items()):
        n = max(1, a["files_num"])
        out.append({
                "top_k": a["top_k"],
                "postprocess_on": a["postprocess_on"],
                "hotwords_strategy": a["hotwords_strategy"],
                "bias_weight_update_cnt": a["bias_weight_update_cnt"],
                "used_file_num": AUDIO_FILE_MAX,
                # round(값, 4)를 통해 소수점 4자리까지 반올림합니다.
                "cer": round(a["total_wrong_char_cnt"]/max(a["total_char_cnt"],1), 4),
                "wer": round(a["total_wrong_morph_cnt"] / max(1, a["total_morph_cnt"]), 4),
                "pn_recall_avg": round(a["pn_recall_sum"] / max(1, a["pn_count"]), 4), # 수정
                "pn_cer_avg": round(a["pn_cer_sum"] / max(1, a["pn_count"]), 4) # 수정
            })

    return out

# ==============================================================================
# 메인 실행 로직
# ==============================================================================
os.makedirs(RESULTS_DIR, exist_ok=True)

now = time.gmtime(time.time()+(9*3600)) #한국 시간
formatted = time.strftime("[%Y%m%d_%H%M]", now)
OUT_ROWS = f"./results/{formatted}_asr_detail.csv"
OUT_SUM  = f"./results/{formatted}_asr_summary.csv"

# total_asr_num=len(HOTWORD_TOPK_SWEEP)

normalizer = TextNormalizer()
mecab = MeCab()
bias_mgr = BiasManager(BIAS_PATH)
transcripts = load_transcripts(TRANSCRIPTS_PATH)
files = glob.glob(os.path.join(AUDIO_FOLDER, "**/*.mp4"), recursive=True)

# ASR 모델 로드
asr = ASR(ASR_MODEL, ASR_DEVICE, ASR_COMPUTE,initial_prompt=KOREAN_ONLY_PROMPT)

rows: List[Dict[str, Any]] = []  # [수정] 결과 데이터를 저장할 리스트

# 2. 실험 루프
for top_k in HOTWORD_TOPK_SWEEP: #hotwords 개수 경우의 수 반복문
    for hotwords_strategy in HOTWORD_STRATEGY_SWEEP: #hotwords 선택 전략 경우의 수 반복문

        for bias_weight_update_cnt in BIAS_WEIGHT_UPDATE_ITERATION_SWEEP: #bias_weight_update 주기 경우의 수 반복문
            # (선택) 각 전략 시작마다 bias 초기화
            if RESET_BIASING_LIST:
                bias_mgr.reset_biasing_list(BIAS_PATH)
            # ✅ 1번 방식: 반복 횟수는 BIAS_WEIGHT_UPDATE_ITERATION_SWEEP
            for repeat in range(bias_weight_update_cnt):

                # ✅ repeat마다 hotwords 새로 샘플링 (1번 방식)
                current_hotwords = bias_mgr.get_weighted_hotwords(top_k, mode=hotwords_strategy)
                for pp_on in POSTPROCESS_SWEEP:
                    pp_str= "ON" if pp_on ==1 else "OFF"
                    print(f"\n[RUN] Top-K: {top_k} | Iteration: {repeat+1}/{bias_weight_update_cnt} | PostProcess: {pp_str}")
                    print(f"hotwords : {current_hotwords}\n")

                    #AUDIO_MAX개만큼의 파일만 뽑아 쓸 때 매번 랜덤하게 파일이 뽑히도록 랜덤 섞기
                    random.shuffle(files) # 리스트의 순서를 무작위로 섞음

                    for audio_path in files[:AUDIO_FILE_MAX]:
                        fname = os.path.basename(audio_path)
                        meta = transcripts.get(fname, {"text": "", "entities": []})

                        # 1) ASR
                        hyp_raw = asr.transcribe(audio_path, "ko", ASR_BEAM, hotwords=current_hotwords)
                        
                        # 2) 후처리
                        if pp_on:
                            hyp_final, replog = postprocess_with_hotwords(
                                hyp_raw, current_hotwords, normalizer,
                                gate=RULE_GATE, tol=RULE_TOL, wratio_th=RULE_WRATIO_TH
                            )
                        else:
                            hyp_final, replog = hyp_raw, []
                        
                        #ref_text 정규화
                        ref_final=normalizer.normalize(meta["text"], False)

                        # ✅ 3) PN 평가: 너가 수정한 4개 리턴 버전 사용
                        pn_recall, pn_cer, hyp_ents, hard_missed_ents = evaluate_proper_nouns(
                            meta.get("entities", []), hyp_final, normalizer, match_th=PN_MATCH_TH, hard_th=HARD_MISS_TH)

                        # 4) Metrics
                        # wrong_char_cnt : 틀린 음절 수, char_cnt : 전체 음절 수
                        # wrong_morph_cnt : 틀린 형태소 수, wrong_morph_cnt : 전체 형태소 수
                        cer, wrong_char_cnt, char_cnt = calculate_cer(ref_final, hyp_final, normalizer)
                        wer, wrong_morph_cnt, morph_cnt, ref_morphs, hyp_morphs = calculate_wer(ref_final, hyp_final, normalizer, mecab,meta["entities"], hyp_ents)

                        # ✅ 1번 방식: hard miss만 학습
                        bias_mgr.add_miss(hard_missed_ents)

                        #pn_recall = pn_recall if pn_recall is not None else 0.0
                        pn_recall_disp = f"{pn_recall:.4f}" if pn_recall is not None else "NA"
                        pn_cer_disp    = f"{pn_cer:.4f}"    if pn_cer    is not None else "NA"

                        # 로그
                        print(
                            f"- file: {os.path.dirname(audio_path).split('/')[-1]}/{fname} | "
                            f"pp_on={pp_on} | cer={cer:.4f} | wer={wer:.4f} | pn_cer={pn_cer_disp} | pn_recall={pn_recall_disp}" # 수정
                        )
                        print(
                            f"ref_text:  [{meta['text']}]\n"
                            f"hyp_raw:   [{hyp_raw}]\n"
                            f"hyp_final: [{hyp_final}]\n"
                            f"ref_pn:    {meta.get('entities', [])}\n"
                            f"hyp_pn:    {hyp_ents}\n"
                            f"hard_miss: {hard_missed_ents}\n"
                        )

                        # 결과 저장(컬럼명 정리 권장)
                        rows.append({
                            "file": f"{os.path.dirname(audio_path).split('/')[-1]}/{fname}",
                            "top_k": top_k,
                            "postprocess_on": int(pp_on),  # 이름 명확히
                            "hotwords_strategy": "random" if hotwords_strategy == 1 else "hybrid",
                            "bias_weight_update_cnt": repeat+1,  # 
                            "hotwords": current_hotwords,

                            "cer": f"{cer:.4f}",
                            "wer": f"{wer:.4f}",
                            "pn_recall": None if pn_recall is None else float(f"{pn_recall:.4f}"), # 수정
                            "pn_cer": None if pn_cer is None else float(f"{pn_cer:.4f}"), # 수정

                            "ref_text": meta["text"],
                            "ref_text_final" : ref_final,
                            "hyp_raw": hyp_raw,
                            "hyp_final": hyp_final,

                            "ref_text_pn": meta.get("entities", []),
                            "hyp_pn": hyp_ents,
                            "hard_missed_pn": hard_missed_ents,

                            "replog": json.dumps(replog, ensure_ascii=False),
                            "wrong_char_cnt":wrong_char_cnt,
                            "char_cnt":char_cnt,
                            "wrong_morph_cnt":wrong_morph_cnt,
                            "morph_cnt":morph_cnt,
                            "ref_morphs": ref_morphs,
                            "hyp_morphs":hyp_morphs
                        })

                # repeat 끝에 학습 반영
                bias_mgr.finalize(bias_weight_update_cnt)

save_results_with_summary(rows, OUT_ROWS)

summary = summarize(rows)
with open(OUT_SUM, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys()))
    w.writeheader()
    w.writerows(summary)

print("\n[DONE] All experiments finished.")
